# APOE region: SuSiE-RSS vs. SuSiE-rRSS PIPs across AD GWAS (data processing)

APOE block (`apoe_block` = chr19:42,346,101–46,842,901, combined region) for every study with both a SuSiE-RSS (`AD_GWAS_rss_baseline_r4_L10`) and a SuSiE-rRSS (`AD_GWAS_rss_ebmix_r4_L10`) fit. For SuSiE-rRSS, the pipeline result is shown: the SER model when the SER fallback was triggered (`serFallbackUsed` in `export/SER_flag.tsv`).

**Output:** `figure_data/FigED2.APOE_PIP_SuSiE_vs_rRSS.rds` — variant × study × method: position, PIP, credible set, SER fallback flag

**Rerun (section at the end):** SuSiE-RSS and SuSiE-rRSS rerun on the APOE block of every study in `29.FigS1.AD_GWAS_QC_Methods` (same inputs for both methods), giving three fits per study: SuSiE-RSS, SuSiE-rRSS before fallback (the adjusted multiple-effect fit) and the full pipeline (its SER model when `R_reliability_flag` is set). Outputs `figure_data/FigED2.APOE_PIP_SuSiE_vs_rRSS_Rerun_Stages.rds` (stages baseline / adjusted / final) and `..._Rerun.rds` (main-table format); the figure notebook uses these when they exist.


In [10]:
source("../../R/utils.R")

data_name <- "FigED2.APOE_PIP_SuSiE_vs_rRSS"
root      <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10"
dirs      <- c("SuSiE-RSS" = "AD_GWAS_rss_baseline_r4_L10", "SuSiE-rRSS" = "AD_GWAS_rss_ebmix_r4_L10")
apoe_block                                                # chr19_42346101_46842901 (R/utils.R)

## studies with an APOE fit under both methods
fit_file <- function(d, s) file.path(root, d, "output/fine_mapping", paste0(s, ".", apoe_block, ".gwas_finemap.rds"))
studies  <- list.files(file.path(root, dirs[["SuSiE-rRSS"]], "output/fine_mapping"),
                       pattern = paste0("\\.", apoe_block, "\\.gwas_finemap\\.rds$")) %>%
  str_remove(paste0("\\.", apoe_block, ".*$"))
studies  <- studies[file.exists(fit_file(dirs[["SuSiE-RSS"]], studies))]
studies

## SER fallback flag per study (same table as Fig 6c / S11a)
ser_flag <- read_tsv(file.path(root, dirs[["SuSiE-rRSS"]], "export/SER_flag.tsv"), show_col_types = FALSE) %>%
  filter(block_id == apoe_block) %>% distinct(study, .keep_all = TRUE) %>%
  transmute(study, ser_fallback = serFallbackUsed %in% TRUE)

## studies with an APOE fit but no row in SER_flag.tsv (their flag is read from the SuSiE-rRSS fit instead)
setdiff(studies, ser_flag$study)

[1] "chr19_42346101_46842901"

[1] "AD_Bellenguez_2022"                     
 [2] "AD_Bellenguez_EADB_2022"                
 [3] "AD_Bellenguez_EADI_2022"                
 [4] "AD_Bellenguez_GRACE_2022"               
 [5] "AD_Bellenguez_main_2026"                
 [6] "AD_Bellenguez_nobiobank_2026"           
 [7] "AD_Bellenguez_noproxy_2026"             
 [8] "AD_Jansen_2021"                         
 [9] "AD_Kunkle_Stage1_2019"                  
[10] "AD_Wightman_Excluding23andMe_2021"      
[11] "AD_Wightman_ExcludingUKBand23andME_2021"
[12] "AD_Wightman_Full_2021"

[1] "AD_Jansen_2021"

In [11]:
## ---- one row per variant x study x method ---------------------------------------
get_fit <- function(f) attributes(readRDS(f))$elementMetadata$susieFit[[1]]

tidy_fit <- function(fit, variants) {
  cs_id <- rep(0L, length(variants))
  if (!is.null(fit$sets$cs)) for (k in seq_along(fit$sets$cs)) cs_id[fit$sets$cs[[k]]] <- k
  tibble(variant = variants,
         pos = as.integer(str_match(variants, "^chr[0-9XY]+[:_]([0-9]+)")[, 2]),
         PIP = as.numeric(fit$pip), cs = cs_id)
}

pip_df <- map_dfr(studies, function(s) {
  flag <- any(ser_flag$ser_fallback[ser_flag$study == s] %in% TRUE)     # no row in SER_flag.tsv -> FALSE
  map_dfr(names(dirs), function(m) {
    fit <- get_fit(fit_file(dirs[[m]], s))
    v   <- colnames(fit$alpha)
    used <- "fit"
    ## study missing from SER_flag.tsv: use the flag stored in the SuSiE-rRSS fit
    if (m == "SuSiE-rRSS" && !s %in% ser_flag$study)
      flag <- isTRUE(fit$R_finite_diagnostics$R_reliability_flag)
    ## SuSiE-rRSS pipeline result: the SER model replaces the fit when the fallback was triggered
    if (m == "SuSiE-rRSS" && flag && !is.null(fit$R_finite_diagnostics$ser_model)) {
      fit <- fit$R_finite_diagnostics$ser_model; used <- "SER model"
    }
    tidy_fit(fit, v) %>% mutate(study = s, method = m, ser_fallback = flag & m == "SuSiE-rRSS", shown = used)
  })
}) %>%
  mutate(study_label = unname(cohort_labels[study]),
         method = factor(method, levels = names(dirs)))

## which fit is drawn; flagged studies whose stored object has no ser_model are drawn from the stored fit
pip_df %>% distinct(study_label, method, ser_fallback, shown) %>% arrange(study_label, method)

save_data(pip_df, data_name)

study_label,method,ser_fallback,shown
<chr>,<fct>,<lgl>,<chr>
Bell.,SuSiE-RSS,FALSE,fit
Bell.,SuSiE-rRSS,TRUE,fit
Bell. EADB,SuSiE-RSS,FALSE,fit
Bell. EADB,SuSiE-rRSS,TRUE,fit
Bell. EADI,SuSiE-RSS,FALSE,fit
Bell. EADI,SuSiE-rRSS,FALSE,fit
Bell. GRACE,SuSiE-RSS,FALSE,fit
Bell. GRACE,SuSiE-rRSS,FALSE,fit
EADB. 2026,SuSiE-RSS,FALSE,fit


## Pre-fallback and final fits for Extended Data Figure 2

Export the saved `multiEffectFit` separately from the selected fit, including the older nested SER format. No pre-fallback result is inferred from a final SER fit; missing adjusted fits remain absent from this companion table.

In [12]:
# Keep the adjusted multiple-effect fit separate from the selected final fit.
# Run after the existing preparation cells, where fit_file(), get_fit() and tidy_fit() are defined.
apoe_stages <- map_dfr(studies, function(s) {
  baseline <- get_fit(fit_file(dirs[["SuSiE-RSS"]], s))
  robust <- get_fit(fit_file(dirs[["SuSiE-rRSS"]], s))
  flag <- if (s %in% ser_flag$study) any(ser_flag$ser_fallback[ser_flag$study==s]) else
    isTRUE(robust$serFallbackUsed) || isTRUE(robust$R_reliability_flag) ||
    isTRUE(robust$R_finite_diagnostics$R_reliability_flag)
  adjusted <- robust$multiEffectFit
  final <- robust
  # Older fit objects store the original adjusted fit and its nested SER alternative.
  if (flag && !is.null(robust$R_finite_diagnostics$ser_model)) {
    if (is.null(adjusted)) adjusted <- robust
    final <- robust$R_finite_diagnostics$ser_model
  }
  if (!flag && is.null(adjusted)) adjusted <- robust
  fits <- list(baseline=baseline, adjusted=adjusted, final=final)
  map_dfr(names(fits), function(stage) {
    f <- fits[[stage]]
    if (is.null(f)) return(tibble()) # Missing fits remain visibly marked in the plotting notebook.
    variants <- colnames(f$alpha)
    if (is.null(variants)) variants <- names(f$pip)
    stopifnot(length(variants)==length(f$pip))
    tidy_fit(f,variants) %>% mutate(study=s,stage=stage,ser_fallback=flag)
  })
})
save_data(apoe_stages,paste0(data_name,"_Stages"))
apoe_stages %>% distinct(study,stage,ser_fallback) %>% count(stage,ser_fallback)


stage,ser_fallback,n
<chr>,<lgl>,<int>
adjusted,FALSE,2
baseline,FALSE,2
baseline,TRUE,10
final,FALSE,2
final,TRUE,10


## Rerun from notebook 29: SuSiE-RSS, SuSiE-rRSS before fallback, full pipeline

Fits from `ADGWAS_QC_methods_APOE/result/<study>.<apoe_block>/` (`29` `[qc_methods] --methods susie_rss susie_rrss`). `susie_rrss.rds` keeps the adjusted fit (`full_fit`), the pipeline result (`fit`: the SER model when flagged) and the flag (`ser_fallback`). Positions and variant IDs come from the unit's `inputs.rds`.

In [13]:
rerun_root  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods_APOE/result"
rerun_units <- basename(list.dirs(rerun_root, recursive = FALSE))
rerun_units

## one notebook-29 fit -> one row per variant (positions / IDs from the unit's inputs, in fit order)
tidy_rerun <- function(fit, inp) {
  v <- inp$sumstat$variant_id
  stopifnot(length(fit$pip) == length(v))
  cs_id <- rep(0L, length(v))
  for (k in seq_along(fit$sets$cs)) cs_id[fit$sets$cs[[k]]] <- k
  tibble(variant = v, pos = inp$sumstat$pos, PIP = as.numeric(fit$pip), cs = cs_id)
}

rerun_stages <- map_dfr(rerun_units, function(u) {
  f    <- function(m) file.path(rerun_root, u, paste0(u, ".", m, ".rds"))
  inp  <- readRDS(f("inputs"))
  rss  <- readRDS(f("susie_rss"))
  rrss <- readRDS(f("susie_rrss"))
  if (!is.null(rss$error) || !is.null(rrss$error)) {
    message(u, ": ", rss$error %||% rrss$error); return(NULL)
  }
  flag <- isTRUE(rrss$ser_fallback)
  fits <- list(baseline = rss$fit, adjusted = rrss$full_fit, final = rrss$fit %||% rrss$full_fit)
  imap_dfr(fits, ~ tidy_rerun(.x, inp) %>%
             mutate(study = inp$study, stage = .y, ser_fallback = flag))
})

## credible sets per study and stage, and the SER fallback flag
rerun_stages %>% group_by(study, stage, ser_fallback) %>%
  summarise(n_cs = n_distinct(cs[cs != 0]), .groups = "drop") %>%
  pivot_wider(names_from = stage, values_from = n_cs) %>% as.data.frame()
rerun_stages %>% distinct(study, ser_fallback) %>% count(ser_fallback)

## main-table format (SuSiE-RSS and the final SuSiE-rRSS fit), as the stored-fit table above
rerun_main <- rerun_stages %>%
  filter(stage != "adjusted") %>%
  transmute(variant, pos, PIP, cs, study,
            method       = factor(if_else(stage == "baseline", "SuSiE-RSS", "SuSiE-rRSS"), levels = c("SuSiE-RSS", "SuSiE-rRSS")),
            ser_fallback = ser_fallback & stage == "final",
            shown        = if_else(stage == "final" & ser_fallback, "SER model", "fit"),
            study_label  = unname(cohort_labels[study]))

save_data(rerun_stages, paste0(data_name, "_Rerun_Stages"))
save_data(rerun_main,   paste0(data_name, "_Rerun"))

[1] "AD_Bellenguez_2022.chr19_42346101_46842901"                     
 [2] "AD_Bellenguez_EADB_2022.chr19_42346101_46842901"                
 [3] "AD_Bellenguez_EADI_2022.chr19_42346101_46842901"                
 [4] "AD_Bellenguez_GRACE_2022.chr19_42346101_46842901"               
 [5] "AD_Bellenguez_main_2026.chr19_42346101_46842901"                
 [6] "AD_Bellenguez_nobiobank_2026.chr19_42346101_46842901"           
 [7] "AD_Bellenguez_noproxy_2026.chr19_42346101_46842901"             
 [8] "AD_Jansen_2021.chr19_42346101_46842901"                         
 [9] "AD_Kunkle_Stage1_2019.chr19_42346101_46842901"                  
[10] "AD_Wightman_Excluding23andMe_2021.chr19_42346101_46842901"      
[11] "AD_Wightman_ExcludingUKBand23andME_2021.chr19_42346101_46842901"
[12] "AD_Wightman_Full_2021.chr19_42346101_46842901"

study,ser_fallback,adjusted,baseline,final
<chr>,<lgl>,<int>,<int>,<int>
AD_Bellenguez_2022,TRUE,3,9,1
AD_Bellenguez_EADB_2022,TRUE,2,3,1
AD_Bellenguez_EADI_2022,FALSE,1,1,1
AD_Bellenguez_GRACE_2022,FALSE,1,2,1
AD_Bellenguez_main_2026,TRUE,2,9,1
AD_Bellenguez_nobiobank_2026,TRUE,2,9,1
AD_Bellenguez_noproxy_2026,TRUE,2,10,1
AD_Jansen_2021,TRUE,6,9,1
AD_Kunkle_Stage1_2019,TRUE,8,10,1


ser_fallback,n
<lgl>,<int>
FALSE,2
TRUE,10


## APOE ε4 and ε2 in the reruns

ε4 (rs429358, chr19:44,908,684) and ε2 (rs7412, chr19:44,908,822) at each rerun stage: **RSS** (SuSiE-RSS), **rRSS** (SuSiE-rRSS before SER fallback) and **Final** (the SuSiE-rRSS pipeline result). For each: the credible set holding the allele (`none` = not in a credible set, `absent` = variant not in the fit), its PIP, and the number of credible sets in the fit; `flagged` = reliability flag set (Final is the SER model).

**Outputs:** `figure_data/FigED2.APOE_PIP_SuSiE_vs_rRSS_Rerun_Alleles` (long: study × stage) and `..._Rerun_Alleles_Wide` (one row per study)

In [14]:
## APOE e4 (rs429358) and e2 (rs7412) in the notebook-29 reruns: credible set and PIP at each stage
alleles   <- c(e4 = 44908684L, e2 = 44908822L)                     # GRCh38
stage_lab <- c(baseline = "RSS", adjusted = "rRSS", final = "Final")
cs_lab    <- function(cs) if_else(is.na(cs), "absent", if_else(cs == 0, "none", paste0("CS", cs)))

## long: one row per study x stage
allele_long <- rerun_stages %>%
  mutate(stage = factor(stage_lab[stage], levels = stage_lab)) %>%
  group_by(study, stage) %>%
  summarise(flagged = any(ser_fallback),
            n_cs    = n_distinct(cs[cs != 0]),
            e4_cs   = cs[match(alleles[["e4"]], pos)], e4_pip = PIP[match(alleles[["e4"]], pos)],
            e2_cs   = cs[match(alleles[["e2"]], pos)], e2_pip = PIP[match(alleles[["e2"]], pos)],
            .groups = "drop") %>%
  mutate(across(c(e4_cs, e2_cs), cs_lab), across(c(e4_pip, e2_pip), ~ round(.x, 3)),
         study = unname(cohort_labels[study])) %>%
  arrange(study, stage)

## wide: one row per study; each stage gives the number of CSs and "CS (PIP)" for e4 and e2
allele_wide <- allele_long %>%
  transmute(study, flagged, stage, n_cs,
            e4 = if_else(is.na(e4_pip), e4_cs, sprintf("%s (%.2f)", e4_cs, e4_pip)),
            e2 = if_else(is.na(e2_pip), e2_cs, sprintf("%s (%.2f)", e2_cs, e2_pip))) %>%
  pivot_wider(names_from = stage, values_from = c(n_cs, e4, e2), names_glue = "{stage}_{.value}") %>%
  select(study, flagged, starts_with("RSS_"), starts_with("rRSS_"), starts_with("Final_"))

allele_long %>% as.data.frame()
allele_wide %>% as.data.frame()
save_data(allele_long, paste0(data_name, "_Rerun_Alleles"))
save_data(allele_wide, paste0(data_name, "_Rerun_Alleles_Wide"))

study,stage,flagged,n_cs,e4_cs,e4_pip,e2_cs,e2_pip
<chr>,<fct>,<lgl>,<int>,<chr>,<dbl>,<chr>,<dbl>
Bell.,RSS,TRUE,9,CS1,1.000,CS2,1.000
Bell.,rRSS,TRUE,3,CS1,1.000,CS2,1.000
Bell.,Final,TRUE,1,CS1,1.000,none,0.000
Bell. EADB,RSS,TRUE,3,CS1,1.000,CS2,1.000
Bell. EADB,rRSS,TRUE,2,CS1,1.000,CS2,1.000
Bell. EADB,Final,TRUE,1,CS1,1.000,none,0.000
Bell. EADI,RSS,FALSE,1,CS1,1.000,none,0.006
Bell. EADI,rRSS,FALSE,1,CS1,1.000,none,0.000
Bell. EADI,Final,FALSE,1,CS1,1.000,none,0.000


study,flagged,RSS_n_cs,RSS_e4,RSS_e2,rRSS_n_cs,rRSS_e4,rRSS_e2,Final_n_cs,Final_e4,Final_e2
<chr>,<lgl>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<chr>,<chr>
Bell.,TRUE,9,CS1 (1.00),CS2 (1.00),3,CS1 (1.00),CS2 (1.00),1,CS1 (1.00),none (0.00)
Bell. EADB,TRUE,3,CS1 (1.00),CS2 (1.00),2,CS1 (1.00),CS2 (1.00),1,CS1 (1.00),none (0.00)
Bell. EADI,FALSE,1,CS1 (1.00),none (0.01),1,CS1 (1.00),none (0.00),1,CS1 (1.00),none (0.00)
Bell. GRACE,FALSE,2,CS1 (1.00),none (0.00),1,CS1 (1.00),none (0.00),1,CS1 (1.00),none (0.00)
EADB. 2026,TRUE,9,CS1 (1.00),CS6 (1.00),2,CS1 (1.00),CS2 (1.00),1,CS1 (1.00),none (0.00)
EADB. 2026 -biobank,TRUE,9,CS1 (1.00),CS2 (1.00),2,CS1 (1.00),CS2 (1.00),1,CS1 (1.00),none (0.00)
EADB. 2026 -proxy,TRUE,10,CS1 (1.00),CS2 (1.00),2,CS1 (1.00),CS2 (1.00),1,CS1 (1.00),none (0.00)
Jansen,TRUE,9,none (0.00),none (0.00),6,CS5 (1.00),CS2 (1.00),1,none (0.00),none (0.00)
Kunkle,TRUE,10,CS1 (1.00),none (0.00),8,CS1 (1.00),none (0.02),1,CS1 (1.00),none (0.00)
